In [4]:
# =========================================================
# PHASE X — LOAD + READ + DISPLAY ADAMSON DATASET
# Purpose:
# 1. Load Adamson dataset
# 2. Inspect shape
# 3. Display metadata
# 4. Show control vs perturbed counts
# 5. Preview genes
# =========================================================

# =========================
# STEP 0 — INSTALL (if needed)
# Uncomment if missing
# !pip install scanpy
# =========================

import scanpy as sc
import pandas as pd
import numpy as np

print("=" * 70)
print("PHASE X — LOAD + READ + DISPLAY ADAMSON DATASET")
print("=" * 70)

# =========================
# STEP 1 — LOAD DATA
# Replace with your Adamson file path
# Example:
#adata = sc.read_h5ad("F:\code\adamson_biolord.h5ad")
# =========================
adamson_path = "adamson_biolord.h5ad"   # CHANGE THIS

adata = sc.read_h5ad(adamson_path)

print("\nDATA LOADED SUCCESSFULLY")

# =========================
# STEP 2 — BASIC SHAPE
# =========================
print("\n" + "=" * 50)
print("DATASET SHAPE")
print("=" * 50)

print("Cells (rows):", adata.n_obs)
print("Genes (cols):", adata.n_vars)
print("Matrix shape :", adata.shape)

# =========================
# STEP 3 — OBS METADATA
# =========================
print("\n" + "=" * 50)
print("OBSERVATION METADATA COLUMNS")
print("=" * 50)

print(adata.obs.columns.tolist())

print("\nSample metadata:")
print(adata.obs.head())

# =========================
# STEP 4 — GENE INFO
# =========================
print("\n" + "=" * 50)
print("GENE / VAR INFO")
print("=" * 50)

print("Var columns:", adata.var.columns.tolist())

print("\nSample genes:")
print(adata.var.head())

# =========================
# STEP 5 — CONTROL VS PERTURBED
# =========================
print("\n" + "=" * 50)
print("CONTROL VS PERTURBED")
print("=" * 50)

# Common Adamson column
if "control" in adata.obs.columns:
    control_cells = (adata.obs["control"] == 1).sum()
    perturbed_cells = (adata.obs["control"] == 0).sum()

    print("Control Cells   :", control_cells)
    print("Perturbed Cells :", perturbed_cells)

# =========================
# STEP 6 — UNIQUE CONDITIONS
# =========================
if "condition" in adata.obs.columns:
    unique_conditions = adata.obs["condition"].astype(str).unique()

    print("\nUnique Conditions:", len(unique_conditions))
    print("Sample Conditions:")
    print(unique_conditions[:20])

# =========================
# STEP 7 — MATRIX PREVIEW
# =========================
print("\n" + "=" * 50)
print("EXPRESSION MATRIX PREVIEW")
print("=" * 50)

# Dense preview
X_preview = adata.X[:5, :10]

# Convert sparse if needed
if hasattr(X_preview, "toarray"):
    X_preview = X_preview.toarray()

preview_df = pd.DataFrame(
    X_preview,
    columns=adata.var_names[:10],
    index=adata.obs_names[:5]
)

print(preview_df)

# =========================
# STEP 8 — QUICK QUALITY CHECK
# =========================
print("\n" + "=" * 50)
print("QUALITY CHECK")
print("=" * 50)

print("Any NaNs in X:", np.isnan(adata.X.toarray()).any() if hasattr(adata.X, "toarray") else np.isnan(adata.X).any())

print("Min expression:",
      adata.X.min() if not hasattr(adata.X, "toarray") else adata.X.min())

print("Max expression:",
      adata.X.max() if not hasattr(adata.X, "toarray") else adata.X.max())

# =========================
# FINAL
# =========================
print("\n" + "=" * 70)
print("PHASE X COMPLETE — ADAMSON DATA READY")
print("=" * 70)

PHASE X — LOAD + READ + DISPLAY ADAMSON DATASET

DATA LOADED SUCCESSFULLY

DATASET SHAPE
Cells (rows): 65899
Genes (cols): 5060
Matrix shape : (65899, 5060)

OBSERVATION METADATA COLUMNS
['condition', 'cell_type', 'dose_val', 'control', 'condition_name', 'split1', 'subgroup1', 'split2', 'subgroup2', 'split3', 'subgroup3', 'split4', 'subgroup4', 'split5', 'subgroup5', 'perturbation']

Sample metadata:
                   condition cell_type dose_val  control  \
cell_barcode                                               
AAACATACACCGAT-1  CREB1+ctrl   K562(?)      1+1        0   
AAACATACAGAGAT-1        ctrl   K562(?)        1        1   
AAACATACCAGAAA-1        ctrl   K562(?)        1        1   
AAACATACGTTGAC-1        ctrl   K562(?)        1        1   
AAACATACTGTTCT-1        ctrl   K562(?)        1        1   

                          condition_name split1      subgroup1 split2  \
cell_barcode                                                            
AAACATACACCGAT-1  K562(?)_CRE

In [5]:
# =========================================================
# PHASE XI — PREPROCESS ADAMSON FOR UNSEEN GENE PIPELINE
# Purpose:
# 1. Separate control vs perturbed
# 2. Clean perturbation labels
# 3. Match each perturbed cell to control baseline
# 4. Build:
#    X_control
#    X_pert
#    perturbation_genes
# =========================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("PHASE XI — ADAMSON PREPROCESSING")
print("=" * 70)

# =========================
# STEP 1 — SPLIT CONTROL / PERTURBED
# =========================
control = adata[adata.obs["control"] == 1].copy()
perturbed = adata[adata.obs["control"] == 0].copy()

print("\nCONTROL SHAPE   :", control.shape)
print("PERTURBED SHAPE :", perturbed.shape)

# =========================
# STEP 2 — CLEAN PERTURBATION LABELS
# Example:
# CREB1+ctrl -> CREB1
# =========================
perturbation_genes = (
    perturbed.obs["condition"]
    .astype(str)
    .str.split("+")
    .str[0]
    .values
)

print("\nUnique perturbation genes:", len(np.unique(perturbation_genes)))
print("Sample genes:", np.unique(perturbation_genes)[:20])

# =========================
# STEP 3 — CONTROL BASELINE
# Mean control profile across all control cells
# =========================
X_ctrl_mean = control.X

if hasattr(X_ctrl_mean, "toarray"):
    X_ctrl_mean = X_ctrl_mean.toarray()

X_ctrl_mean = X_ctrl_mean.mean(axis=0)

print("\nGlobal control baseline shape:", X_ctrl_mean.shape)

# =========================
# STEP 4 — BUILD MATCHED MATRICES
# For each perturbed cell:
# X_control = same baseline
# X_pert    = actual perturbed expression
# =========================
X_pert = perturbed.X

if hasattr(X_pert, "toarray"):
    X_pert = X_pert.toarray()

# Repeat control baseline for each perturbed cell
X_control = np.repeat(
    X_ctrl_mean.reshape(1, -1),
    X_pert.shape[0],
    axis=0
)

print("\nX_control:", X_control.shape)
print("X_pert   :", X_pert.shape)

# =========================
# STEP 5 — QUALITY CHECK
# =========================
print("\n" + "=" * 50)
print("QUALITY CHECK")
print("=" * 50)

print("Gene dimensions match:", X_control.shape[1] == X_pert.shape[1])
print("Cell counts match    :", X_control.shape[0] == X_pert.shape[0])

print("\nExpression ranges:")
print("Control Min:", X_control.min(), "| Max:", X_control.max())
print("Perturb Min:", X_pert.min(), "| Max:", X_pert.max())

# =========================
# STEP 6 — EFFECT MATRIX
# Perturbation effect:
# delta = perturbed - control
# =========================
P = X_pert - X_control

print("\nP (Perturbation Effect Matrix):", P.shape)

# =========================
# STEP 7 — GENE NAME MAP
# =========================
gene_ids = adata.var_names.tolist()

if "gene_name" in adata.var.columns:
    gene_symbols = adata.var["gene_name"].astype(str).tolist()
else:
    gene_symbols = gene_ids

print("\nGenes mapped:", len(gene_symbols))
print("Sample gene symbols:", gene_symbols[:20])

# =========================
# STEP 8 — SAVE CORE OBJECTS
# =========================
print("\n" + "=" * 70)
print("CORE VARIABLES READY:")
print("=" * 70)

print("control")
print("perturbed")
print("X_control")
print("X_pert")
print("P")
print("perturbation_genes")
print("gene_symbols")

# =========================
# FINAL
# =========================
print("\nPHASE XI COMPLETE — READY FOR:")
print("1. Unseen gene holdout")
print("2. Gene embeddings")
print("3. Geneformer-lite")
print("=" * 70)

PHASE XI — ADAMSON PREPROCESSING

CONTROL SHAPE   : (24263, 5060)
PERTURBED SHAPE : (41636, 5060)

Unique perturbation genes: 81
Sample genes: ['AARS' 'ARHGAP22' 'ASCC3' 'ATP5B' 'BHLHE40' 'CAD' 'CARS' 'CCND3' 'CHERP'
 'COPB1' 'COPZ1' 'CREB1' 'DAD1' 'DARS' 'DDIT3' 'DDOST' 'DDRGK1' 'DERL2'
 'DHDDS' 'DNAJC19']

Global control baseline shape: (5060,)

X_control: (41636, 5060)
X_pert   : (41636, 5060)

QUALITY CHECK
Gene dimensions match: True
Cell counts match    : True

Expression ranges:
Control Min: 1.3798398e-05 | Max: 5.218205
Perturb Min: 0.0 | Max: 7.978193

P (Perturbation Effect Matrix): (41636, 5060)

Genes mapped: 5060
Sample gene symbols: ['AP006222.2', 'RP11-54O7.16', 'RP11-54O7.1', 'RP11-54O7.3', 'SAMD11', 'HES4', 'ISG15', 'TTLL10', 'TNFRSF18', 'TNFRSF4', 'SDF4', 'CCNL2', 'RP4-758J18.13', 'ATAD3C', 'RP11-345P4.9', 'MIB2', 'MMP23B', 'CALML6', 'RP5-892K4.1', 'RP11-181G12.2']

CORE VARIABLES READY:
control
perturbed
X_control
X_pert
P
perturbation_genes
gene_symbols

PHASE XI CO

In [6]:
# =========================================================
# PHASE XII — NORMALIZATION + FEATURE PREP
# Purpose:
# 1. Standardize control / perturbed matrices
# 2. Normalize perturbation effect
# 3. Remove zero-variance genes
# 4. Prepare ML-ready matrices
#
# Output:
# Xc_norm
# Xp_norm
# P_norm
# valid_gene_mask
# =========================================================

import numpy as np
from sklearn.preprocessing import StandardScaler

print("=" * 70)
print("PHASE XII — NORMALIZATION + FEATURE PREP")
print("=" * 70)

# =========================
# STEP 1 — REMOVE ZERO-VARIANCE GENES
# Important:
# Genes with no variance add noise
# =========================
gene_variance = X_pert.var(axis=0)

valid_gene_mask = gene_variance > 1e-8

print("\nOriginal genes :", X_pert.shape[1])
print("Valid genes    :", valid_gene_mask.sum())
print("Removed genes  :", (~valid_gene_mask).sum())

# Filter
X_control_f = X_control[:, valid_gene_mask]
X_pert_f    = X_pert[:, valid_gene_mask]

print("\nFiltered shapes:")
print("X_control_f:", X_control_f.shape)
print("X_pert_f   :", X_pert_f.shape)

# =========================
# STEP 2 — STANDARDIZE USING TRAINING DISTRIBUTION
# Fit on perturbed for realistic biological spread
# =========================
scaler_control = StandardScaler()
scaler_pert    = StandardScaler()

Xc_norm = scaler_control.fit_transform(X_control_f)
Xp_norm = scaler_pert.fit_transform(X_pert_f)

print("\nNormalized:")
print("Xc_norm:", Xc_norm.shape)
print("Xp_norm:", Xp_norm.shape)

# =========================
# STEP 3 — NORMALIZED EFFECT MATRIX
# =========================
P_norm = Xp_norm - Xc_norm

print("P_norm :", P_norm.shape)

# =========================
# STEP 4 — SANITY CHECK
# =========================
print("\n" + "=" * 50)
print("POST-NORMALIZATION CHECK")
print("=" * 50)

print("Control mean (approx):", round(Xc_norm.mean(), 6))
print("Control std  (approx):", round(Xc_norm.std(), 6))

print("Perturb mean (approx):", round(Xp_norm.mean(), 6))
print("Perturb std  (approx):", round(Xp_norm.std(), 6))

print("\nAny NaNs:")
print("Xc_norm:", np.isnan(Xc_norm).any())
print("Xp_norm:", np.isnan(Xp_norm).any())
print("P_norm :", np.isnan(P_norm).any())

# =========================
# STEP 5 — UPDATE GENE SYMBOLS
# =========================
gene_symbols_filtered = np.array(gene_symbols)[valid_gene_mask]

print("\nFiltered gene symbols:", len(gene_symbols_filtered))
print("Sample:", gene_symbols_filtered[:20])

# =========================
# STEP 6 — OPTIONAL CLIPPING
# Prevent extreme outliers
# =========================
clip_value = 5

Xc_norm = np.clip(Xc_norm, -clip_value, clip_value)
Xp_norm = np.clip(Xp_norm, -clip_value, clip_value)
P_norm  = np.clip(P_norm,  -clip_value, clip_value)

print(f"\nClipped to range: [-{clip_value}, {clip_value}]")

# =========================
# FINAL OUTPUT
# =========================
print("\n" + "=" * 70)
print("PHASE XII COMPLETE — NORMALIZED DATA READY")
print("=" * 70)

print("Xc_norm")
print("Xp_norm")
print("P_norm")
print("valid_gene_mask")
print("gene_symbols_filtered")

print("\nNEXT:")
print("1. Unseen gene holdout")
print("2. Biological embeddings")
print("3. Geneformer-lite training")
print("=" * 70)

PHASE XII — NORMALIZATION + FEATURE PREP

Original genes : 5060
Valid genes    : 5058
Removed genes  : 2

Filtered shapes:
X_control_f: (41636, 5058)
X_pert_f   : (41636, 5058)

Normalized:
Xc_norm: (41636, 5058)
Xp_norm: (41636, 5058)
P_norm : (41636, 5058)

POST-NORMALIZATION CHECK
Control mean (approx): 0.0
Control std  (approx): 0.0
Perturb mean (approx): -0.0
Perturb std  (approx): 1.0

Any NaNs:
Xc_norm: False
Xp_norm: False
P_norm : False

Filtered gene symbols: 5058
Sample: ['AP006222.2' 'RP11-54O7.16' 'RP11-54O7.1' 'RP11-54O7.3' 'SAMD11' 'HES4'
 'ISG15' 'TTLL10' 'TNFRSF18' 'TNFRSF4' 'SDF4' 'CCNL2' 'RP4-758J18.13'
 'ATAD3C' 'RP11-345P4.9' 'MIB2' 'MMP23B' 'CALML6' 'RP5-892K4.1'
 'RP11-181G12.2']

Clipped to range: [-5, 5]

PHASE XII COMPLETE — NORMALIZED DATA READY
Xc_norm
Xp_norm
P_norm
valid_gene_mask
gene_symbols_filtered

NEXT:
1. Unseen gene holdout
2. Biological embeddings
3. Geneformer-lite training


In [7]:
# =========================================================
# PHASE XIII — GENEFORMER-STYLE TRAIN / VAL / TEST SPLIT
# Purpose:
# TRUE unseen-gene split (OOD)
#
# Train:
# Seen perturbation genes
#
# Test:
# One completely unseen gene
#
# Validation:
# Random from seen train genes
# =========================================================

import numpy as np
from sklearn.model_selection import train_test_split

print("=" * 70)
print("PHASE XIII — GENEFORMER-STYLE SPLIT")
print("=" * 70)

# =========================
# STEP 1 — CHOOSE UNSEEN GENE
# Change anytime
# =========================
unseen_gene = "CREB1"

print("\nChosen unseen gene:", unseen_gene)

# =========================
# STEP 2 — VERIFY
# =========================
all_genes = np.unique(perturbation_genes)

if unseen_gene not in all_genes:
    raise ValueError(f"{unseen_gene} not found.")

print("Total unique genes:", len(all_genes))

# =========================
# STEP 3 — HOLDOUT
# =========================
train_mask = perturbation_genes != unseen_gene
test_mask  = perturbation_genes == unseen_gene

print("\nCells:")
print("Train+Val:", train_mask.sum())
print("Test     :", test_mask.sum())

# =========================
# STEP 4 — TRAIN / VAL SPLIT
# ONLY from seen genes
# =========================
train_indices = np.where(train_mask)[0]

train_idx, val_idx = train_test_split(
    train_indices,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

test_idx = np.where(test_mask)[0]

print("\nSplit sizes:")
print("Train:", len(train_idx))
print("Val  :", len(val_idx))
print("Test :", len(test_idx))

# =========================
# STEP 5 — BUILD MATRICES
# Using normalized matrices
# =========================
Xc_train = Xc_norm[train_idx]
Xp_train = Xp_norm[train_idx]

Xc_val = Xc_norm[val_idx]
Xp_val = Xp_norm[val_idx]

Xc_test = Xc_norm[test_idx]
Xp_test = Xp_norm[test_idx]

# =========================
# STEP 6 — LABELS
# =========================
train_genes = perturbation_genes[train_idx]
val_genes   = perturbation_genes[val_idx]
test_genes  = perturbation_genes[test_idx]

# =========================
# STEP 7 — QUALITY CHECK
# =========================
print("\n" + "=" * 50)
print("QUALITY CHECK")
print("=" * 50)

print("Train unique genes:", len(np.unique(train_genes)))
print("Val unique genes  :", len(np.unique(val_genes)))
print("Test unique genes :", len(np.unique(test_genes)))

print("\nTest genes:")
print(np.unique(test_genes))

# Leakage check
leakage = unseen_gene in np.unique(train_genes)

print("\nLeakage present:", leakage)

# =========================
# STEP 8 — SHAPES
# =========================
print("\n" + "=" * 50)
print("FINAL MATRICES")
print("=" * 50)

print("Xc_train:", Xc_train.shape)
print("Xp_train:", Xp_train.shape)

print("Xc_val  :", Xc_val.shape)
print("Xp_val  :", Xp_val.shape)

print("Xc_test :", Xc_test.shape)
print("Xp_test :", Xp_test.shape)

# =========================
# STEP 9 — BASELINE CONTROL
# =========================
# Useful later for zero-shot baseline
control_baseline_test = Xc_test.copy()

# =========================
# FINAL
# =========================
print("\n" + "=" * 70)
print("PHASE XIII COMPLETE — GENEFORMER SPLIT READY")
print("=" * 70)

print("train_idx")
print("val_idx")
print("test_idx")

print("Xc_train / Xp_train")
print("Xc_val / Xp_val")
print("Xc_test / Xp_test")

print("\nNEXT:")
print("1. Build biological gene embeddings")
print("2. Train Geneformer-lite")
print("3. Zero-shot unseen evaluation")
print("=" * 70)

PHASE XIII — GENEFORMER-STYLE SPLIT

Chosen unseen gene: CREB1
Total unique genes: 81

Cells:
Train+Val: 41164
Test     : 472

Split sizes:
Train: 37047
Val  : 4117
Test : 472

QUALITY CHECK
Train unique genes: 80
Val unique genes  : 80
Test unique genes : 1

Test genes:
['CREB1']

Leakage present: False

FINAL MATRICES
Xc_train: (37047, 5058)
Xp_train: (37047, 5058)
Xc_val  : (4117, 5058)
Xp_val  : (4117, 5058)
Xc_test : (472, 5058)
Xp_test : (472, 5058)

PHASE XIII COMPLETE — GENEFORMER SPLIT READY
train_idx
val_idx
test_idx
Xc_train / Xp_train
Xc_val / Xp_val
Xc_test / Xp_test

NEXT:
1. Build biological gene embeddings
2. Train Geneformer-lite
3. Zero-shot unseen evaluation


In [8]:
# =========================================================
# PHASE XIII — GENEFORMER TOKENIZATION + SPLIT
# Purpose:
# Prepare Adamson for actual Geneformer-style training
#
# Geneformer expects:
# Rank-based gene tokens per cell
# NOT raw expression vectors
#
# Core idea:
# Each cell -> ranked gene list (highest expressed genes first)
# =========================================================

import numpy as np
from sklearn.model_selection import train_test_split

print("=" * 70)
print("PHASE XIII — GENEFORMER TOKENIZATION + SPLIT")
print("=" * 70)

# =========================
# STEP 1 — UNSEEN GENE
# =========================
unseen_gene = "CREB1"

print("\nUnseen gene:", unseen_gene)

# =========================
# STEP 2 — HOLDOUT
# =========================
train_mask = perturbation_genes != unseen_gene
test_mask  = perturbation_genes == unseen_gene

train_indices_all = np.where(train_mask)[0]
test_idx = np.where(test_mask)[0]

train_idx, val_idx = train_test_split(
    train_indices_all,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

print("\nCells:")
print("Train:", len(train_idx))
print("Val  :", len(val_idx))
print("Test :", len(test_idx))

# =========================
# STEP 3 — GENE TOKEN MAP
# Each gene gets token ID
# 0 reserved for padding
# =========================
gene_token_dict = {
    gene_symbols_filtered[i]: i + 1
    for i in range(len(gene_symbols_filtered))
}

print("\nVocabulary size:", len(gene_token_dict))

# =========================
# STEP 4 — TOKENIZER
# Convert expression profile -> ranked gene tokens
# Highest expression genes first
# =========================
def tokenize_cell(expression_vector, gene_names, max_len=2048):
    # Rank descending
    ranked_idx = np.argsort(expression_vector)[::-1]

    ranked_genes = gene_names[ranked_idx]

    tokens = [
        gene_token_dict[g]
        for g in ranked_genes
        if g in gene_token_dict
    ]

    # Truncate
    tokens = tokens[:max_len]

    return tokens

# =========================
# STEP 5 — BUILD TOKEN MATRICES
# Using perturbed state (Geneformer standard)
# =========================
def build_token_dataset(X, gene_names, max_len=2048):
    tokenized = []

    for i in range(X.shape[0]):
        tokens = tokenize_cell(X[i], gene_names, max_len=max_len)
        tokenized.append(tokens)

        if i % 5000 == 0:
            print(f"Processed {i}/{X.shape[0]} cells...")

    return tokenized

# =========================
# STEP 6 — TOKENIZE
# =========================
print("\nTokenizing train...")
train_tokens = build_token_dataset(
    Xp_norm[train_idx],
    gene_symbols_filtered
)

print("\nTokenizing val...")
val_tokens = build_token_dataset(
    Xp_norm[val_idx],
    gene_symbols_filtered
)

print("\nTokenizing test...")
test_tokens = build_token_dataset(
    Xp_norm[test_idx],
    gene_symbols_filtered
)

# =========================
# STEP 7 — PAD
# =========================
max_len = 2048

def pad_tokens(token_lists, max_len=2048):
    padded = np.zeros((len(token_lists), max_len), dtype=np.int32)

    for i, tokens in enumerate(token_lists):
        length = min(len(tokens), max_len)
        padded[i, :length] = tokens[:length]

    return padded

X_train_tokens = pad_tokens(train_tokens, max_len)
X_val_tokens   = pad_tokens(val_tokens, max_len)
X_test_tokens  = pad_tokens(test_tokens, max_len)

# =========================
# STEP 8 — LABELS
# Predict perturbation gene token/class
# =========================
unique_train_genes = sorted(np.unique(train_genes))

perturb_token_map = {
    gene: i
    for i, gene in enumerate(unique_train_genes)
}

y_train = np.array([perturb_token_map[g] for g in train_genes])

# Validation labels
y_val = np.array([perturb_token_map[g] for g in val_genes])

# Test:
# unseen -> -1
y_test = np.array([
    perturb_token_map[g] if g in perturb_token_map else -1
    for g in test_genes
])

# =========================
# STEP 9 — SUMMARY
# =========================
print("\n" + "=" * 50)
print("TOKENIZED DATASET SUMMARY")
print("=" * 50)

print("X_train_tokens:", X_train_tokens.shape)
print("X_val_tokens  :", X_val_tokens.shape)
print("X_test_tokens :", X_test_tokens.shape)

print("\nTrain classes:", len(np.unique(y_train)))
print("Val classes  :", len(np.unique(y_val)))
print("Test unseen labels:", np.unique(y_test))

# =========================
# STEP 10 — SAMPLE
# =========================
print("\nSample tokenized cell:")
print(X_train_tokens[0][:50])

# =========================
# FINAL
# =========================
print("\n" + "=" * 70)
print("PHASE XIII COMPLETE — GENEFORMER INPUT READY")
print("=" * 70)

print("X_train_tokens")
print("X_val_tokens")
print("X_test_tokens")

print("y_train")
print("y_val")
print("y_test")

print("\nNEXT:")
print("1. Build Geneformer transformer")
print("2. Pretrain / finetune")
print("3. Zero-shot unseen gene evaluation")
print("=" * 70)

PHASE XIII — GENEFORMER TOKENIZATION + SPLIT

Unseen gene: CREB1

Cells:
Train: 37047
Val  : 4117
Test : 472

Vocabulary size: 5058

Tokenizing train...
Processed 0/37047 cells...
Processed 5000/37047 cells...
Processed 10000/37047 cells...
Processed 15000/37047 cells...
Processed 20000/37047 cells...
Processed 25000/37047 cells...
Processed 30000/37047 cells...
Processed 35000/37047 cells...

Tokenizing val...
Processed 0/4117 cells...

Tokenizing test...
Processed 0/472 cells...

TOKENIZED DATASET SUMMARY
X_train_tokens: (37047, 2048)
X_val_tokens  : (4117, 2048)
X_test_tokens : (472, 2048)

Train classes: 80
Val classes  : 80
Test unseen labels: [-1]

Sample tokenized cell:
[ 255 4880   71 2385 2919  180 3334 2358 1692 4526 2454 3197 3276 3051
 1626 5026  752 4172 4217 4493 2485  676 4762 4189 2597  247 4374 2785
 3030 1661 3803 3426  414 1658 3634 2342 4454 2893 4753 3752  604  913
 3452 1046 1413 4709  600  825 4838  214]

PHASE XIII COMPLETE — GENEFORMER INPUT READY
X_train_token

In [ ]:
# =========================================================
# PHASE XIV — MEMORY-SAFE GENEFORMER MODEL
# Optimized for CPU / low RAM
#
# Key fixes:
# 1. Shorter token length (512)
# 2. Smaller embedding dim
# 3. Fewer transformer layers
# 4. Smaller batch
# =========================================================

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================
# STEP 1 — TRUNCATE TOKENS
# Reduce 2048 -> 512
# =========================
max_len_small = 512

X_train_small = X_train_tokens[:, :max_len_small]
X_val_small   = X_val_tokens[:, :max_len_small]
X_test_small  = X_test_tokens[:, :max_len_small]

print("Reduced token shape:")
print("Train:", X_train_small.shape)
print("Val  :", X_val_small.shape)
print("Test :", X_test_small.shape)

# =========================
# DATASET
# =========================
class GeneformerDataset(Dataset):
    def __init__(self, X_tokens, y):
        self.X = torch.tensor(X_tokens, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# =========================
# DATALOADERS
# =========================
batch_size = 8   # CPU safe

train_loader = DataLoader(
    GeneformerDataset(X_train_small, y_train),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    GeneformerDataset(X_val_small, y_val),
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    GeneformerDataset(X_test_small, np.zeros(len(X_test_small))),
    batch_size=batch_size,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

# =========================
# POSITIONAL ENCODING
# =========================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) *
            (-np.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# =========================
# MEMORY-SAFE MODEL
# =========================
class GeneformerLite(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_classes,
        embed_dim=64,
        num_heads=4,
        num_layers=2,
        ff_dim=128,
        max_len=512,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size + 1,
            embed_dim,
            padding_idx=0
        )

        self.pos_encoder = PositionalEncoding(embed_dim, max_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # Token embedding
        x = self.embedding(x)

        # Positional
        x = self.pos_encoder(x)

        # Transformer
        x = self.transformer(x)

        # Mean pooling
        pooled = x.mean(dim=1)

        # Classification
        logits = self.classifier(pooled)

        return logits, pooled

# =========================
# INIT MODEL
# =========================
vocab_size = len(gene_token_dict)
num_classes = len(np.unique(y_train))

model = GeneformerLite(
    vocab_size=vocab_size,
    num_classes=num_classes
).to(device)

print(model)

# =========================
# LOSS
# =========================
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

# =========================
# TRAIN LOOP
# =========================
epochs = 10
best_val_acc = 0

for epoch in range(epochs):

    # -----------------
    # TRAIN
    # -----------------
    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for X, y in train_loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()

        logits, _ = model(X)

        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        preds = logits.argmax(dim=1)

        train_correct += (preds == y).sum().item()
        train_total += y.size(0)

    train_acc = train_correct / train_total

    # -----------------
    # VAL
    # -----------------
    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)

            logits, _ = model(X)

            preds = logits.argmax(dim=1)

            val_correct += (preds == y).sum().item()
            val_total += y.size(0)

    val_acc = val_correct / val_total

    # Save best
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_geneformer_lite_adamson.pt")

    print(
        f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train Loss: {train_loss/len(train_loader):.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

print("\nBEST VAL ACC:", best_val_acc)
print("Saved: best_geneformer_lite_adamson.pt")

Device: cpu
Reduced token shape:
Train: (37047, 512)
Val  : (4117, 512)
Test : (472, 512)
Train batches: 4631
Val batches: 515
GeneformerLite(
  (embedding): Embedding(5059, 64, padding_idx=0)
  (pos_encoder): PositionalEncoding()
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (classifier): Sequential(
    (0): Linear(